# InsertionSort (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time/InsertionSort_java.ipynb)

Chapter 12 — Running time. Runnable notebook of `InsertionSort`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time/InsertionSort_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time/InsertionSort_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the Java code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Insertion sort: InsertionSort

**Input:** an array $a[1..n]$ of $n$ natural numbers. **Output:** the array $a$ sorted in non-decreasing order. If $t(i)$ is the number of shifts at iteration $i$, then $S(n) = 6n - 4 + 3\sum_{i=2}^{n} t(i)$; in the worst case (decreasing array) $S(n) = \frac{3n^2+9n-8}{2} = \Theta(n^2)$.

Pseudo-code: section 12.3 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

In [2]:
%%writefile InsertionSort.java
public class InsertionSort
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

// "[a[from], ..., a[to-1]]"
static String fmt(int[] a, int from, int to)
{
    StringBuilder s = new StringBuilder("[");
    for (int i = from; i < to; i++)
    {
        if (i > from) s.append(", ");
        s.append(a[i]);
    }
    return s.append("]").toString();
}

    // InsertionSort(a, n)
    static void insertionSort(int[] a, boolean trace)
    {
        int n = a.length;
        long prev = 0;                     // only for the trace
        int i = 1; cnt[2]++;               // i := 2 in the pseudo-code
        cnt[3]++;
        while (i < n)                      // i <= n in the pseudo-code
        {
            int key = a[i]; cnt[4]++;
            int j = i - 1; cnt[5]++;
            cnt[6]++;                      // the while condition is counted at every evaluation
            while (j >= 0 && a[j] > key)   // j >= 1 and a[j] > key in the pseudo-code
            {
                a[j + 1] = a[j]; cnt[7]++;
                j = j - 1; cnt[8]++;
                cnt[6]++;
            }
            a[j + 1] = key; cnt[10]++;
            i = i + 1; cnt[11]++;
            cnt[3]++;
            if (trace)                     // trace: after the increment, i is the pseudo-code index of the iteration just finished
            {
                System.out.println(String.format("%2d | %3d | %4d | %s", i, key, cnt[7] - prev, fmt(a, 0, n)));
                prev = cnt[7];
            }
        }
    }

    static long stepsOf(int[] a)           // works on a copy of a
    {
        reset();
        insertionSort(a.clone(), false);
        return total();
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: a[1..6] = (5, 2, 4, 6, 1, 3)
        int[] a = {5, 2, 4, 6, 1, 3};
        int n = a.length;
        System.out.printf("a = %s\n", fmt(a, 0, n));
        reset();
        System.out.printf(" i | key | t(i) | array after the iteration\n");
        insertionSort(a, true);
        System.out.printf("sorted: %s\n", fmt(a, 0, n));
        printSteps(2, 3, 4, 5, 6, 7, 8, 10, 11);
        long t = cnt[7];
        System.out.printf("sum t(i) = %d, S(n) = 6n - 4 + 3 sum t(i) = %d\n", t, 6 * n - 4 + 3 * t);

        // Worst case: decreasing array, t(i) = i - 1
        System.out.printf("\n");
        int[] d = {6, 5, 4, 3, 2, 1};
        System.out.printf("a = %s (decreasing): S = %d = (3n^2 + 9n - 8)/2 = %d\n", fmt(d, 0, n), stepsOf(d), (3 * n * n + 9 * n - 8) / 2);
        System.out.printf("sum t(i) = %d = n(n-1)/2 = %d\n", cnt[7], n * (n - 1) / 2);

        System.out.printf("\n");
        System.out.printf("Decreasing array (n, ..., 2, 1):\n");
        System.out.printf("  n | S(n) | (3n^2 + 9n - 8)/2\n");
        for (int k = 1; k <= 8; k++)
        {
            int[] b = new int[k];
            for (int i = 0; i < k; i++) b[i] = k - i;
            System.out.printf("%3d | %4d | %d\n", k, stepsOf(b), (3 * k * k + 9 * k - 8) / 2);
        }
    }
}

Writing InsertionSort.java


In [3]:
!javac InsertionSort.java && java InsertionSort

a = [5, 2, 4, 6, 1, 3]
 i | key | t(i) | array after the iteration
 2 |   2 |    1 | [2, 5, 4, 6, 1, 3]
 3 |   4 |    1 | [2, 4, 5, 6, 1, 3]
 4 |   6 |    0 | [2, 4, 5, 6, 1, 3]
 5 |   1 |    4 | [1, 2, 4, 5, 6, 3]
 6 |   3 |    3 | [1, 2, 3, 4, 5, 6]
sorted: [1, 2, 3, 4, 5, 6]
Line | Executions
   2 |          1
   3 |          6
   4 |          5
   5 |          5
   6 |         14
   7 |          9
   8 |          9
  10 |          5
  11 |          5
Total steps S = 59
sum t(i) = 9, S(n) = 6n - 4 + 3 sum t(i) = 59

a = [6, 5, 4, 3, 2, 1] (decreasing): S = 77 = (3n^2 + 9n - 8)/2 = 77
sum t(i) = 15 = n(n-1)/2 = 15

Decreasing array (n, ..., 2, 1):
  n | S(n) | (3n^2 + 9n - 8)/2
  1 |    2 | 2
  2 |   11 | 11
  3 |   23 | 23
  4 |   38 | 38
  5 |   56 | 56
  6 |   77 | 77
  7 |  101 | 101
  8 |  128 | 128


**Try it yourself.** Change the array and check the formula $6n - 4 + 3\sum t(i)$.

In [4]:
%%writefile ProvaInsertionSort.java
public class ProvaInsertionSort
{
    public static void main(String[] args)
    {
        int[] a = {8, 3, 5, 1, 9, 2, 7};      // <-- change the array
        int n = a.length;
        InsertionSort.reset();
        System.out.printf(" i | key | t(i) | array after the iteration\n");
        InsertionSort.insertionSort(a, true);
        InsertionSort.printSteps(2, 3, 4, 5, 6, 7, 8, 10, 11);
        System.out.printf("6n - 4 + 3 sum t(i) = %d\n", 6 * n - 4 + 3 * InsertionSort.cnt[7]);
    }
}

Writing ProvaInsertionSort.java


In [5]:
!javac ProvaInsertionSort.java && java ProvaInsertionSort

 i | key | t(i) | array after the iteration
 2 |   3 |    1 | [3, 8, 5, 1, 9, 2, 7]
 3 |   5 |    1 | [3, 5, 8, 1, 9, 2, 7]
 4 |   1 |    3 | [1, 3, 5, 8, 9, 2, 7]
 5 |   9 |    0 | [1, 3, 5, 8, 9, 2, 7]
 6 |   2 |    4 | [1, 2, 3, 5, 8, 9, 7]
 7 |   7 |    2 | [1, 2, 3, 5, 7, 8, 9]
Line | Executions
   2 |          1
   3 |          7
   4 |          6
   5 |          6
   6 |         17
   7 |         11
   8 |         11
  10 |          6
  11 |          6
Total steps S = 71
6n - 4 + 3 sum t(i) = 71


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).